In [2]:
# ==========================================
# 🎙️ AI PODCAST ANALYZER
# Whisper + BART + Gradio
# ==========================================

!pip -q install -U openai-whisper transformers sentencepiece gradio

import whisper
import torch
import gradio as gr

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

# ------------------------------------------
# 1. Load Whisper
# ------------------------------------------

print("Loading Whisper model...")

speech_model = whisper.load_model("base")

# ------------------------------------------
# 2. Load Summarization Model
# ------------------------------------------

print("Loading summarization model...")

MODEL_NAME = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

summary_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

print("Models loaded successfully!")

# ------------------------------------------
# 3. Summarization Function
# ------------------------------------------

def generate_summary(text):

    # Keep input within model limit
    text = text[:4000]

    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )

    with torch.no_grad():

        summary_ids = summary_model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=150,
            min_length=40,
            num_beams=4,
            early_stopping=True
        )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary


# ------------------------------------------
# 4. Keyword Extraction
# ------------------------------------------

def extract_keywords(text):

    stop_words = {
        "the", "and", "is", "in", "to", "of",
        "a", "for", "on", "with", "that",
        "this", "are", "was", "it", "as",
        "be", "from", "an", "or", "we",
        "you", "they", "i", "he", "she",
        "have", "has", "had", "will"
    }

    words = (
        text.lower()
        .replace(".", "")
        .replace(",", "")
        .replace("!", "")
        .replace("?", "")
        .split()
    )

    keywords = []

    for word in words:

        word = word.strip("!?;:\"'()[]")

        if len(word) > 4 and word not in stop_words:

            if word not in keywords:
                keywords.append(word)

        if len(keywords) == 10:
            break

    return ", ".join(keywords)


# ------------------------------------------
# 5. Main Podcast Analyzer
# ------------------------------------------

def analyze_podcast(audio):

    if audio is None:
        return "Please upload or record a podcast."

    try:

        print("Transcribing audio...")

        # Speech to text
        result = speech_model.transcribe(audio)

        transcript = result["text"].strip()

        if not transcript:
            return "No speech detected in the audio."

        print("Generating summary...")

        # Generate summary
        summary = generate_summary(transcript)

        # Extract keywords
        keywords = extract_keywords(transcript)

        # Final result
        output = f"""
🎙️ AI PODCAST ANALYSIS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📝 TRANSCRIPT
{transcript}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📌 SUMMARY
{summary}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔑 KEYWORDS
{keywords}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

        return output

    except Exception as e:

        return f"Error: {str(e)}"


# ------------------------------------------
# 6. Gradio Interface
# ------------------------------------------

with gr.Blocks(
    title="AI Podcast Analyzer"
) as app:

    gr.Markdown("""
    # 🎙️ AI Podcast Analyzer

    ### Analyze podcast audio using Artificial Intelligence

    **Features:**
    - 📝 Speech-to-text transcription
    - 📌 Automatic summarization
    - 🔑 Keyword extraction
    """)

    audio_input = gr.Audio(
        sources=["upload", "microphone"],
        type="filepath",
        label="🎤 Upload or Record Podcast"
    )

    analyze_button = gr.Button(
        "🔍 Analyze Podcast"
    )

    result_output = gr.Textbox(
        label="📊 Podcast Analysis",
        lines=25
    )

    analyze_button.click(
        fn=analyze_podcast,
        inputs=audio_input,
        outputs=result_output
    )

    gr.Markdown("""
    **Technologies:** Python | Whisper | BART | NLP | Gradio
    """)


# ------------------------------------------
# 7. Launch
# ------------------------------------------

app.launch(share=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 1.5 MB/s eta 0:00:00
Loading Whisper model...
Loading summarization model...


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Models loaded successfully!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://300ed8386af63a13ad.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
